# Task 1 — World Happiness Choropleth + Ranked Country List (Altair)

Builds the two Task 1 visualizations for the **Drivers of Happiness** site:

1. **Choropleth map** of `life_evaluation` (Cantril ladder, 0–10) with a **year slider** (2006–2025) and a **▶ Play button** in the exported page that animates through the years with a cross-fade
2. **Ranked country list** — horizontal bars, sorted from happiest down, with its own year slider

Both are exported as standalone HTML files at the end, ready to drop into `charts/` and iframe into `index.html`. Both use the site's theme (fonts, ink colors) and a shared red→yellow→green scale (green = happier).

**Running in Google Colab:** upload this notebook, run all cells — the data-loading cell will prompt you to upload `whr_2006_2025_cleaned.csv` (it lives in `charts/data/` in the repo). Running locally inside the repo works too and skips the upload. Note: the Play button and cross-fade only exist in the exported `task1_choropleth.html`, not in the inline Colab preview.

In [ ]:
%pip install -q "altair>=5.0" country_converter

## Load the data

Reads from the repo if the notebook is run locally; otherwise prompts for an upload in Colab.

In [ ]:
import os
import pandas as pd

CSV_PATH = "charts/data/whr_2006_2025_cleaned.csv"  # path when run from the repo root

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
elif os.path.exists("whr_2006_2025_cleaned.csv"):
    df_raw = pd.read_csv("whr_2006_2025_cleaned.csv")
else:
    from google.colab import files  # only exists in Colab
    print("Upload whr_2006_2025_cleaned.csv")
    uploaded = files.upload()
    df_raw = pd.read_csv(next(iter(uploaded)))

print(df_raw.shape)
df_raw[["year", "country", "country_code", "region", "life_evaluation"]].head()

## Prepare the map data

The world topojson identifies countries by **ISO 3166-1 numeric** id, but the CSV has **ISO3 letter codes** (`AFG`, `FIN`, …). `country_converter` bridges the two. We also compute each country's **happiness rank within its year** for the tooltip.

Kosovo (`XKX`) has no official ISO numeric code and no clean id in the world atlas, so it can't render on the map — it gets reported below so we can mention it in the write-up rather than lose it silently.

In [ ]:
import country_converter as coco

cols = ["year", "country", "country_code", "region", "life_evaluation"]
df = df_raw[cols].dropna(subset=["life_evaluation"]).copy()

cc = coco.CountryConverter()
iso3_codes = sorted(df["country_code"].unique())
numeric = cc.convert(names=iso3_codes, src="ISO3", to="ISOnumeric", not_found=None)
iso3_to_num = dict(zip(iso3_codes, numeric))
iso3_to_num["XKX"] = None  # Kosovo — see note above

df["id"] = df["country_code"].map(iso3_to_num)
no_id = df.loc[df["id"].isna(), "country"].unique()
print("No map id (will not render on the map):", list(no_id))
df = df.dropna(subset=["id"]).astype({"id": int})

df["rank"] = (
    df.groupby("year")["life_evaluation"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print(f"{df['country'].nunique()} countries, {len(df)} country-year rows")
df.head()

## Check the join against the world atlas

We use the **50m-resolution** world atlas (rather than the coarser 110m) so small countries like Singapore, Malta, and Bahrain actually have polygons to color. This cell fetches it and cleans two quirks:

- **"Ashmore and Cartier Is." shares id `036` with Australia** — Vega-Lite's lookup silently resolves `036` to the tiny islands, leaving mainland Australia uncolored. We keep only the first feature per id.
- **Antarctica** is dropped (no data — just a gray slab eating vertical space).

It then matches the atlas id format (ids are zero-padded strings like `"004"`) and reports any country in our data with no shape on the map — so nothing disappears silently. The cleaned shapes are embedded directly into the chart, so the exported HTML also works offline.

In [ ]:
import json
import urllib.request

TOPO_URL = "https://cdn.jsdelivr.net/npm/world-atlas@2/countries-50m.json"

with urllib.request.urlopen(TOPO_URL) as r:
    topo = json.load(r)

# Drop Antarctica and keep only the first feature per id (fixes the
# Australia / Ashmore and Cartier Is. id collision on 036).
seen, cleaned = set(), []
for g in topo["objects"]["countries"]["geometries"]:
    gid = g.get("id")
    if gid == "010":
        continue
    if gid is not None and gid in seen:
        print("Dropping duplicate-id feature:", g["properties"]["name"])
        continue
    seen.add(gid)
    cleaned.append(g)
topo["objects"]["countries"]["geometries"] = cleaned

atlas_ids = {str(g["id"]) for g in cleaned if g.get("id") is not None}

def to_map_id(n):
    padded = str(int(n)).zfill(3)
    return padded if padded in atlas_ids else str(int(n))

df["map_id"] = df["id"].apply(to_map_id)

missing = df.loc[~df["map_id"].isin(atlas_ids), "country"].unique()
print("In our data but not on the 50m map:", list(missing) or "none")

## Choropleth with year slider

Design choices:
- **Diverging red → yellow → green scale** (green = happier), fixed domain `[1, 8]` so colors mean the same thing in every year as you drag the slider. *Caveat:* red–green is the hardest pairing for colorblind viewers (~4% of men); the ranked list and tooltips carry the same information, which is the mitigation. If the team ever wants a fully CVD-safe version, swap the scheme to `"redyellowblue"` — one line.
- A neutral **gray base layer** underneath shows countries with no data for the selected year, instead of leaving holes.
- **Equal Earth projection** with a soft ocean tint and graticule lines, so the map reads as designed rather than default.
- **Site theme** — same font stack and ink colors as `styles.css`, applied through `site_theme()` so both charts match the website.
- A **`key` encoding on country** keeps each country bound to the same SVG node across year changes — that's what lets the exported page cross-fade colors when animating.
- Tooltip carries country, region, year, score, and rank.

In [ ]:
import altair as alt

SITE_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif'
INK, INK_SOFT, LINE = "#2f3542", "#6b7280", "#ece4d4"

def site_theme(chart):
    """Match the chart chrome to the website's styles.css."""
    return (
        chart.configure(font=SITE_FONT, background="white")
        .configure_view(stroke=None)
        .configure_title(
            anchor="start", color=INK, fontSize=19, fontWeight=700,
            subtitleColor=INK_SOFT, subtitleFontSize=13, subtitlePadding=6, offset=16,
        )
        .configure_axis(
            labelColor=INK_SOFT, titleColor=INK, labelFontSize=11, titleFontSize=12,
            gridColor=LINE, domainColor=LINE, tickColor=LINE,
        )
        .configure_legend(
            titleColor=INK, labelColor=INK_SOFT, titleFontSize=12, labelFontSize=11,
            gradientLength=280, gradientThickness=10,
        )
    )

# Embed the cleaned shapes inline (rather than alt.topo_feature(TOPO_URL, ...))
# so the duplicate-id fix above actually reaches the chart.
countries = alt.Data(
    values=topo,
    format=alt.TopoDataFormat(type="topojson", feature="countries"),
)

YEAR_MIN, YEAR_MAX = int(df["year"].min()), int(df["year"].max())

year_sel = alt.param(
    name="year_sel",
    value=YEAR_MAX,
    bind=alt.binding_range(min=YEAR_MIN, max=YEAR_MAX, step=1, name="Year:  "),
)

color_scale = alt.Scale(scheme="redyellowgreen", domain=[1, 8])

tooltip = [
    alt.Tooltip("country:N", title="Country"),
    alt.Tooltip("region:N", title="Region"),
    alt.Tooltip("year:Q", title="Year", format="d"),
    alt.Tooltip("life_evaluation:Q", title="Life evaluation", format=".2f"),
    alt.Tooltip("rank:Q", title="Rank"),
]

ocean = alt.Chart(alt.sphere()).mark_geoshape(fill="#eef3f6")

graticule = alt.Chart(alt.graticule(step=[30, 30])).mark_geoshape(
    filled=False, stroke="white", strokeWidth=0.5
)

base = alt.Chart(countries).mark_geoshape(
    fill="#dad7cf", stroke="white", strokeWidth=0.4
)

choro = (
    alt.Chart(df)
    .mark_geoshape(stroke="white", strokeWidth=0.4)
    .transform_filter(alt.datum.year == year_sel)
    .transform_lookup(
        lookup="map_id",
        from_=alt.LookupData(countries, key="id"),
        as_="geo",
    )
    .transform_calculate(
        geometry="datum.geo.geometry",
        type="datum.geo.type",
    )
    .encode(
        color=alt.Color(
            "life_evaluation:Q",
            scale=color_scale,
            legend=alt.Legend(
                title="Life evaluation (0–10)", format=".1f",
                orient="bottom", direction="horizontal",
            ),
        ),
        key="country:N",  # object constancy — enables the cross-fade when animating
        tooltip=tooltip,
    )
)

map_chart = site_theme(
    (ocean + graticule + base + choro)
    .add_params(year_sel)
    .project(type="equalEarth")
    .properties(
        width=850,
        height=430,
        title=alt.Title(
            "How happy is the world?",
            subtitle="Average life evaluation (Cantril ladder, 0–10). Drag the slider to change year; gray countries have no data that year.",
        ),
    )
)

map_chart

## Ranked country list

The Task 1 "list" component as a horizontal bar chart — every country for the selected year, sorted happiest-first, colored on the same scale as the map so the two read as one system. It's tall by design; the iframe on the site scrolls it.

In [ ]:
year_sel_list = alt.param(
    name="year_sel_list",
    value=YEAR_MAX,
    bind=alt.binding_range(min=YEAR_MIN, max=YEAR_MAX, step=1, name="Year:  "),
)

list_chart = site_theme(
    alt.Chart(df)
    .transform_filter(alt.datum.year == year_sel_list)
    .mark_bar(size=10, cornerRadiusEnd=3)
    .encode(
        x=alt.X(
            "life_evaluation:Q",
            title="Life evaluation (0–10)",
            scale=alt.Scale(domain=[0, 8]),
            axis=alt.Axis(orient="top"),  # visible without scrolling to the bottom
        ),
        y=alt.Y("country:N", sort="-x", title=None),
        color=alt.Color("life_evaluation:Q", scale=color_scale, legend=None),
        tooltip=tooltip,
    )
    .add_params(year_sel_list)
    .properties(
        width=620,
        height=alt.Step(16),
        title=alt.Title(
            "Countries ranked by happiness",
            subtitle="Sorted by life evaluation for the selected year",
        ),
    )
)

list_chart

## Export standalone HTML for the website

The **map** is exported through a small custom HTML template instead of `chart.save()`, which adds:

- a themed **▶ Play button** that steps the year signal 2006 → 2025 every 0.5 s (click again to pause; it restarts from 2006 when it finishes),
- a **CSS cross-fade** on country fills, so colors melt between years instead of snapping (the `key` encoding above keeps each country on the same SVG node, which is what makes this work),
- slider styling that matches the website.

The **list** is a plain `chart.save()`. Both files are fully self-contained. In Colab this cell also triggers browser downloads.

In [ ]:
MAP_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>World happiness choropleth</title>
<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
<style>
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
      "Helvetica Neue", Arial, sans-serif;
    margin: 0;
    padding: 14px 18px;
    background: #ffffff;
  }
  #controls { display: flex; align-items: center; gap: 12px; margin-bottom: 10px; }
  #play {
    border: none;
    background: #f6b93b;
    color: #ffffff;
    font: inherit;
    font-weight: 600;
    font-size: 14px;
    padding: 8px 20px;
    border-radius: 999px;
    cursor: pointer;
    transition: background 0.15s ease;
  }
  #play:hover { background: #e58e26; }
  #controls span { color: #6b7280; font-size: 13px; }
  /* cross-fade country fills between years */
  #vis svg path { transition: fill 0.45s ease; }
  /* theme the vega slider row */
  #vis .vega-bindings { margin-top: 10px; color: #6b7280; font-size: 13px; }
  #vis .vega-bindings input[type="range"] {
    accent-color: #f6b93b;
    width: 260px;
    vertical-align: middle;
  }
</style>
</head>
<body>
<div id="controls">
  <button id="play" type="button">&#9654;&nbsp; Play</button>
  <span>animates 2006 &rarr; 2025, or drag the slider below the map</span>
</div>
<div id="vis"></div>
<script>
  const spec = __SPEC__;
  const YEAR_MIN = __YMIN__, YEAR_MAX = __YMAX__, STEP_MS = 500;

  vegaEmbed("#vis", spec, { actions: false }).then(({ view }) => {
    const btn = document.getElementById("play");
    let timer = null;

    function stop() {
      clearInterval(timer);
      timer = null;
      btn.innerHTML = "&#9654;&nbsp; Play";
    }

    btn.addEventListener("click", () => {
      if (timer) { stop(); return; }
      if (view.signal("year_sel") >= YEAR_MAX) {
        view.signal("year_sel", YEAR_MIN).runAsync();  // restart from the beginning
      }
      btn.innerHTML = "&#10074;&#10074;&nbsp; Pause";
      timer = setInterval(() => {
        const y = view.signal("year_sel") + 1;
        if (y > YEAR_MAX) { stop(); return; }
        view.signal("year_sel", y).runAsync();
      }, STEP_MS);
    });
  });
</script>
</body>
</html>
"""

map_html = (
    MAP_TEMPLATE
    .replace("__SPEC__", map_chart.to_json(indent=None))
    .replace("__YMIN__", str(YEAR_MIN))
    .replace("__YMAX__", str(YEAR_MAX))
)
with open("task1_choropleth.html", "w") as f:
    f.write(map_html)

list_chart.save("task1_country_list.html", embed_options={"actions": False})
print("Saved task1_choropleth.html and task1_country_list.html")

try:
    from google.colab import files
    files.download("task1_choropleth.html")
    files.download("task1_country_list.html")
except ImportError:
    pass  # running locally — files are next to the notebook

## Embedding into the site

Move the two exported files into `charts/`, then replace the Task 1 placeholders in `index.html` with:

```html
<iframe src="charts/task1_choropleth.html" class="viz-frame"
        loading="lazy" style="min-height: 640px"
        title="World happiness choropleth"></iframe>

<iframe src="charts/task1_country_list.html" class="viz-frame"
        loading="lazy" style="min-height: 640px"
        title="Countries ranked by happiness"></iframe>
```

Notes:
- Data and country shapes are embedded in the files (map ~2.3 MB, list ~0.5 MB — fine for Vercel; `loading="lazy"` keeps them off the critical path). The map page pulls the vega/vega-lite libraries from the jsDelivr CDN, so it needs internet — always true on the deployed site.
- The map iframe needs a little extra height for the play button row + slider; ~640px fits.
- The country list is intentionally tall; the iframe scrolls it. Adjust `min-height` to taste.
- Kosovo has data but no map polygon (no ISO numeric id) — worth a one-line footnote on the site.